In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, median_absolute_error

#load in the cleaned test and training datasets
test_df = pd.read_csv("cleaned_test.csv")
train_df = pd.read_csv("cleaned_training.csv")

feature_cols = [
    'LivingArea', 
    'BedroomsTotal', 
    'BathroomsTotalInteger',
    'LotSizeSquareFeet', 
    'zip_median_price', 
    'city_median_price',
    'bed_bath_ratio', 
    'property_age', 
    'district_median_price'
]

X_train = train_df[feature_cols].copy()
y_train = train_df['ClosePrice'].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df['ClosePrice'].copy()

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")

X_train shape: (71099, 9) | y_train shape: (71099,)
X_test shape:  (12784, 9)  | y_test shape:  (12784,)


In [17]:
# finds average percentage error between the true and predicted values
def mape(y_true, y_pred):
 y_true = np.array(y_true)
 y_pred = np.array(y_pred)
 mask = y_true != 0
 ape = np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]) * 100
 return np.mean(ape)

#finds the median percentage error between the true and predicted values (after sorting)
def mdape(y_true, y_pred):
 y_true = np.array(y_true)
 y_pred = np.array(y_pred)
 mask = y_true != 0
 ape = np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]) * 100
 return np.median(ape)

def regression_report(model, y_true, y_pred):
 return {
  'Model' : model,
 'R²': r2_score(y_true, y_pred),
 'MAPE': mape(y_true, y_pred),
 'MDAPE': mdape(y_true, y_pred),
 }

In [18]:
results = []

# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
results.append(regression_report('Linear Regression', y_test, y_pred_lr))

# Decision Tree
decision_tree = DecisionTreeRegressor()
decision_tree.fit(X_train, y_train)
y_pred_dt = decision_tree.predict(X_test)
results.append(regression_report('Decision Tree', y_test, y_pred_dt))

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
results.append(regression_report('Random Forest', y_test, y_pred_rf))

# XGBoost Tuned - from Week 7
tuned_xgb = XGBRegressor(
    max_depth=9,
    learning_rate=0.1,
    n_estimators=300,
    random_state=42
)

tuned_xgb.fit(X_train, y_train)
y_pred_xgb = tuned_xgb.predict(X_test)
results.append(regression_report('XGBoost Tuned', y_test, y_pred_xgb))

results_df = pd.DataFrame(results).sort_values(by='R²', ascending=False).reset_index(drop=True)
print('Metrics Comparison:')
display(results_df)

Metrics Comparison:


,Model,R²,MAPE,MDAPE
0,XGBoost Tuned,0.872436,13.641219,8.748570
1,Random Forest,0.865517,14.072713,8.802520
2,Linear Regression,0.799203,24.543565,17.900594
3,Decision Tree,0.735416,19.274797,12.195122


- XGBoost Tuned
    - lowest MAPE score indicating that the model's predictions are off by ~13.6% on average
    - lowest MDAPE score indicating half the predictions have an error lower than 8.7%
- Linear Regression
    - highest MAPE and MDAPE score -> more errors in prediction

Price Band Analysis

In [19]:
#Create results DataFrame
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted rf": y_pred_rf,
    "Predicted dt": y_pred_dt,
    "Predicted lr": y_pred_lr,
    "Predicted xgb": y_pred_xgb
})

#Assign Price Bands
price_bands = [
    "Under $500k",
    "$500k-$1M",
    "$1M-$2M",
    "Over $2M"
]

results["Price Band"] = pd.cut(
    results["Actual"],
    bins=[0, 500000, 1000000, 2000000,np.inf],
    labels=price_bands
)

def calc_mape(actual, predicted):
    pct_errors = np.abs((actual - predicted) / actual) * 100
    return round(np.mean(pct_errors), 2)


def calc_mdape(actual, predicted):
    pct_errors = np.abs((actual - predicted) / actual) * 100
    return round(np.median(pct_errors), 2)

#Calculate MAPE
mape_results = []
for band in ["Under $500k", "$500k-$1M", "$1M-$2M", "Over $2M"]:

    subset = results[results["Price Band"] == band]

    mape_results.append({
        "Price Band": band,
        "Count": len(subset),
        "RF MAPE%": calc_mape(subset["Actual"], subset["Predicted rf"]),
        "DT MAPE%": calc_mape(subset["Actual"], subset["Predicted dt"]),
        "LR MAPE%": calc_mape(subset["Actual"], subset["Predicted lr"]),
        "XGB MAPE%": calc_mape(subset["Actual"], subset["Predicted xgb"])
    })

mape_df = pd.DataFrame(mape_results)

print("MAPE by Price Band:")
mape_df

MAPE by Price Band:


,Price Band,Count,RF MAPE%,DT MAPE%,LR MAPE%,XGB MAPE%
0,Under $500k,1809,20.62,25.81,44.91,20.62
1,$500k-$1M,5327,11.05,14.77,23.82,10.64
2,$1M-$2M,3946,13.55,19.84,18.19,12.92
3,Over $2M,1702,17.77,25.12,19.88,17.27


- XGBoost has the lowest MAPE across all price bands, followed closely by Random Forest
- **500K–1M** is the best-performing band for all models — XGBoost achieves its lowest error here at 10.64% MAPE
- **Under 500K** has the highest error (XGBoost/RF: 20.62%), due to fewer training examples in this price range
- **1M–2M** shows moderate error — XGBoost 12.92%, RF 13.55%
- **Over 2M** sees higher error (XGBoost 17.27%, RF 17.77%), as luxury homes have unique features not captured in MLS data
- Linear Regression struggles most under 500K (44.91%) but becomes more competitive above 1M

In [20]:
mdape_results = []

for band in ["Under $500k", "$500k-$1M", "$1M-$2M", "Over $2M"]:

    subset = results[results["Price Band"] == band]

    mdape_results.append({
        "Price Band": band,
        "RF MdAPE%": calc_mdape(subset["Actual"], subset["Predicted rf"]),
        "DT MdAPE%": calc_mdape(subset["Actual"], subset["Predicted dt"]),
        "LR MdAPE%": calc_mdape(subset["Actual"], subset["Predicted lr"]),
        "XGB MdAPE%": calc_mdape(subset["Actual"], subset["Predicted xgb"])
    })

mdape_df = pd.DataFrame(mdape_results)

print("MdAPE by Price Band:")
mdape_df

MdAPE by Price Band:


,Price Band,RF MdAPE%,DT MdAPE%,LR MdAPE%,XGB MdAPE%
0,Under $500k,9.22,12.68,35.40,9.64
1,$500k-$1M,6.85,9.09,18.68,6.98
2,$1M-$2M,9.94,14.29,13.48,9.52
3,Over $2M,14.46,20.00,16.69,13.76


- XGBoost and Random Forest have the lowest MdAPE across all price bands
- **$500K–$1M** is the best performing band
    - RF: 6.85%, XGBoost: 6.98%
- **Under $500K** shows higher error (RF: 9.22%, XGBoost: 9.64%) due to fewer training examples in this price range
- **$1M–$2M** sees moderate error, XGBoost: 9.52%, RF: 9.94% 
    - Linear Regression (13.48%) outperforms Decision Tree (13.98%) in this tier
- **Over $2M** has the highest error across all models (XGBoost: 13.76%, RF: 14.46%) as luxury homes have unique features not captured in MLS data

In [21]:
metrics_summary = mape_df.merge(
    mdape_df,
    on="Price Band"
)

print("Metrics by Price Band:")
display(metrics_summary)

metrics_summary.to_csv("metrics_summary.csv", index=False)

Metrics by Price Band:


,Price Band,Count,RF MAPE%,DT MAPE%,LR MAPE%,XGB MAPE%,RF MdAPE%,DT MdAPE%,LR MdAPE%,XGB MdAPE%
0,Under $500k,1809,20.62,25.81,44.91,20.62,9.22,12.68,35.40,9.64
1,$500k-$1M,5327,11.05,14.77,23.82,10.64,6.85,9.09,18.68,6.98
2,$1M-$2M,3946,13.55,19.84,18.19,12.92,9.94,14.29,13.48,9.52
3,Over $2M,1702,17.77,25.12,19.88,17.27,14.46,20.00,16.69,13.76


#### Summary:

- **500K–1M** is the best-performing band across all models and metrics, where most of the training data exists
- Error increases at both extremes: under 500K due to fewer training examples, and over 2M due to unique luxury features not captured in MLS data

1. **XGBoost** — Best overall model
    - Lowest MAPE (10.64%) and MdAPE (6.98%) in the 500K–1M band
    - Most consistent performance across all price tiers

2. **Random Forest** — Close second
    - Slightly better MdAPE in the under 500K band (9.22% vs. XGBoost 9.64%)

3. **Decision Tree** — High variance
    - Most unstable across price bands (MAPE ranges from 14.73% to 24.95%)

4. **Linear Regression** — Weakest overall
    - Struggles most under 500K (MAPE: 44.91%, MdAPE: 35.40%)
    - Becomes more competitive above 1M